# 01. GAPS ETF 데이터 수집

제12회 GAPS ETF 188개 종목의 OHLCV + 기술지표 수집  
저장 위치: `내 드라이브/gaps_competition/data/`

In [ ]:
# ── 패키지 설치 ──────────────────────────────────────────────────────────────
!pip install -q FinanceDataReader pykrx pandas-ta tqdm

In [ ]:
# ── Google Drive 마운트 ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/졸업프로젝트/GAPS_대회/data'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'저장 경로: {DRIVE_PATH}')

In [ ]:
# ── ETF 유니버스 정의 (etf_universe.py 인라인) ─────────────────────────────────
# Colab에서 실행 시 아래 리스트를 직접 사용
# 로컬에서는 from src.etf_universe import ETF_LIST 사용 가능

ETF_LIST = [
    # FX 및 원자재 (위험)
    {'ticker': '411060', 'name': 'ACE KRX금현물',              'is_risk': True,  'category': 'fx_commodity'},
    {'ticker': '144600', 'name': 'KODEX 은선물(H)',              'is_risk': True,  'category': 'fx_commodity'},
    {'ticker': '132030', 'name': 'KODEX 골드선물(H)',            'is_risk': True,  'category': 'fx_commodity'},
    {'ticker': '261220', 'name': 'KODEX WTI원유선물(H)',         'is_risk': True,  'category': 'fx_commodity'},
    {'ticker': '261240', 'name': 'KODEX 미국달러선물',           'is_risk': True,  'category': 'fx_commodity'},
    {'ticker': '292560', 'name': 'TIGER 일본엔선물',             'is_risk': True,  'category': 'fx_commodity'},
    {'ticker': '271060', 'name': 'KODEX 3대농산물선물(H)',       'is_risk': True,  'category': 'fx_commodity'},
    # 해외주식_지수 (위험)
    {'ticker': '360750', 'name': 'TIGER 미국S&P500',                    'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '379800', 'name': 'KODEX 미국S&P500',                    'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '133690', 'name': 'TIGER 미국나스닥100',                  'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '379810', 'name': 'KODEX 미국나스닥100',                  'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥',      'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '360200', 'name': 'ACE 미국S&P500',                      'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '367380', 'name': 'ACE 미국나스닥100',                    'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '379780', 'name': 'RISE 미국S&P500',                     'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '371160', 'name': 'TIGER 차이나항셍테크',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '449180', 'name': 'KODEX 미국S&P500(H)',                  'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '251350', 'name': 'KODEX 선진국MSCI World',               'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '448290', 'name': 'TIGER 미국S&P500TR(H)',                'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '241180', 'name': 'TIGER 일본니케이225',                  'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '453810', 'name': 'KODEX 인도Nifty50',                   'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '453870', 'name': 'TIGER 인도니프티50',                   'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '245710', 'name': 'ACE 베트남VN30(합성)',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '448300', 'name': 'TIGER 미국나스닥100TR(H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '283580', 'name': 'KODEX 차이나CSI300',                   'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '200250', 'name': 'KIWOOM 인도Nifty50(합성)',             'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '414780', 'name': 'TIGER 차이나과창판STAR50(합성)',        'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '372330', 'name': 'KODEX 차이나항셍테크',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '192090', 'name': 'TIGER 차이나CSI300',                   'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '219480', 'name': 'KODEX 미국S&P500선물(H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '245340', 'name': 'TIGER 미국다우존스30',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '168580', 'name': 'ACE 중국본토CSI300',                   'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '488500', 'name': 'TIGER 미국S&P500동일가중',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '304940', 'name': 'KODEX 미국나스닥100선물(H)',            'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '143850', 'name': 'TIGER 미국S&P500선물(H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '195980', 'name': 'PLUS 신흥국MSCI(합성 H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '101280', 'name': 'KODEX 일본TOPIX100',                   'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '099140', 'name': 'KODEX 차이나H',                        'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '245360', 'name': 'TIGER 차이나HSCEI',                    'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '238720', 'name': 'ACE 일본Nikkei225(H)',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '195930', 'name': 'TIGER 유로스탁스50(합성 H)',            'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '256440', 'name': 'ACE 인도네시아MSCI(합성)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '280930', 'name': 'KODEX 미국러셀2000(H)',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '195970', 'name': 'PLUS 선진국MSCI(합성 H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '250730', 'name': 'RISE 차이나HSCEI(H)',                  'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '291130', 'name': 'ACE 멕시코MSCI(합성)',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '195920', 'name': 'TIGER 일본TOPIX(합성 H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '310080', 'name': 'RISE 중국MSCI China(H)',               'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '105010', 'name': 'TIGER 라틴35',                         'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '379790', 'name': 'RISE 유로스탁스50(H)',                 'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '411860', 'name': 'KIWOOM 독일DAX',                       'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '499150', 'name': 'SOL 미국S&P500엔화노출(H)',             'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '189400', 'name': 'PLUS 글로벌MSCI(합성 H)',              'is_risk': True,  'category': 'overseas_index'},
    {'ticker': '261920', 'name': 'ACE 필리핀MSCI(합성)',                 'is_risk': True,  'category': 'overseas_index'},
    # 해외주식_섹터 (위험)
    {'ticker': '381170', 'name': 'TIGER 미국테크TOP10 INDXX',            'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '458730', 'name': 'TIGER 미국배당다우존스',                'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '487230', 'name': 'KODEX 미국AI전력핵심인프라',            'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '371460', 'name': 'TIGER 차이나전기차SOLACTIVE',           'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '457480', 'name': 'ACE 테슬라밸류체인액티브',              'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '456600', 'name': 'TIME 글로벌AI인공지능액티브',           'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '465580', 'name': 'ACE 미국빅테크TOP7 Plus',               'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '446770', 'name': 'ACE 글로벌반도체TOP4 Plus',             'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '446720', 'name': 'SOL 미국배당다우존스',                  'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '497570', 'name': 'TIGER 미국필라델피아AI반도체나스닥',    'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '402970', 'name': 'ACE 미국배당다우존스',                  'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '390390', 'name': 'KODEX 미국반도체',                      'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '452360', 'name': 'SOL 미국배당다우존스(H)',               'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '182480', 'name': 'TIGER 미국MSCI리츠(합성 H)',            'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '429000', 'name': 'TIGER 미국S&P500배당귀족',              'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '463680', 'name': 'KODEX 미국S&P500테크놀로지',            'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '453640', 'name': 'KODEX 미국S&P500헬스케어',              'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '200030', 'name': 'KODEX 미국S&P500산업재(합성)',          'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '453650', 'name': 'KODEX 미국S&P500금융',                  'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '218420', 'name': 'KODEX 미국S&P500에너지(합성)',          'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '352560', 'name': 'KODEX 미국부동산리츠(H)',               'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '493420', 'name': 'SOL 미국배당다우존스TR',                'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '287180', 'name': 'PLUS 미국나스닥테크',                   'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '245350', 'name': 'TIGER 유로스탁스배당30',                'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '494410', 'name': 'PLUS 미국S&P500성장주',                'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '460660', 'name': 'RISE 미국S&P배당킹',                    'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '185680', 'name': 'KODEX 미국S&P바이오(합성)',             'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '453630', 'name': 'KODEX 미국S&P500필수소비재',            'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '463640', 'name': 'KODEX 미국S&P500유틸리티',              'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '453660', 'name': 'KODEX 미국S&P500경기소비재',            'is_risk': True,  'category': 'overseas_sector'},
    {'ticker': '463690', 'name': 'KODEX 미국S&P500커뮤니케이션',          'is_risk': True,  'category': 'overseas_sector'},
    # 국내주식_지수 (위험)
    {'ticker': '069500', 'name': 'KODEX 200',           'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '229200', 'name': 'KODEX 코스닥150',     'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '102110', 'name': 'TIGER 200',           'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '278530', 'name': 'KODEX 200TR',         'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '310970', 'name': 'TIGER MSCI Korea TR', 'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '148020', 'name': 'RISE 200',            'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '232080', 'name': 'TIGER 코스닥150',     'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '278540', 'name': 'KODEX MSCI Korea TR', 'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '294400', 'name': 'KIWOOM 200TR',        'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '152100', 'name': 'PLUS 200',            'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '226490', 'name': 'KODEX 코스피',        'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '361580', 'name': 'RISE 200TR',          'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '277630', 'name': 'TIGER 코스피',        'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '295040', 'name': 'SOL 200TR',           'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '302450', 'name': 'RISE 코스피',         'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '328370', 'name': 'PLUS 코스피TR',       'is_risk': True,  'category': 'domestic_index'},
    {'ticker': '156080', 'name': 'KODEX MSCI Korea',    'is_risk': True,  'category': 'domestic_index'},
    # 국내주식_섹터 (위험)
    {'ticker': '091160', 'name': 'KODEX 반도체',               'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '161510', 'name': 'PLUS 고배당주',               'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '102780', 'name': 'KODEX 삼성그룹',              'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '395160', 'name': 'KODEX AI반도체',              'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '466920', 'name': 'SOL 조선TOP3플러스',          'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '305720', 'name': 'KODEX 2차전지산업',           'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '449450', 'name': 'PLUS K방산',                  'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '487240', 'name': 'KODEX AI전력핵심설비',        'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '395270', 'name': 'HANARO Fn K-반도체',          'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '315930', 'name': 'KODEX Top5PlusTR',            'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '305540', 'name': 'TIGER 2차전지테마',           'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '494670', 'name': 'TIGER 조선TOP10',             'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '445290', 'name': 'KODEX 로봇액티브',            'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '455850', 'name': 'SOL AI반도체소부장',           'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '466940', 'name': 'TIGER 은행고배당플러스TOP10',  'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '463250', 'name': 'TIGER K방산&우주',            'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139260', 'name': 'TIGER 200 IT',                'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '462010', 'name': 'TIGER 2차전지소재Fn',         'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '434730', 'name': 'HANARO 원자력iSelect',        'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '091180', 'name': 'KODEX 자동차',                'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '091170', 'name': 'KODEX 은행',                  'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139230', 'name': 'TIGER 200 중공업',            'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '325010', 'name': 'KODEX 성장주',                'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '363580', 'name': 'KODEX 200IT TR',              'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139220', 'name': 'TIGER 200 건설',              'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139270', 'name': 'TIGER 200 금융',              'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '227540', 'name': 'TIGER 200 헬스케어',          'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '147970', 'name': 'TIGER 모멘텀',                'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139250', 'name': 'TIGER 200 에너지화학',        'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '174350', 'name': 'TIGER 로우볼',                'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '227550', 'name': 'TIGER 200 산업재',            'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '227570', 'name': 'TIGER 우량가치',              'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139290', 'name': 'TIGER 200 경기소비재',        'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '139240', 'name': 'TIGER 200 철강소재',          'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '315270', 'name': 'TIGER 200커뮤니케이션서비스', 'is_risk': True,  'category': 'domestic_sector'},
    {'ticker': '227560', 'name': 'TIGER 200 생활소비재',        'is_risk': True,  'category': 'domestic_sector'},
    # 금리연계형/초단기채권 (안전)
    {'ticker': '459580', 'name': 'KODEX CD금리액티브(합성)',         'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '488770', 'name': 'KODEX 머니마켓액티브',             'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '423160', 'name': 'KODEX KOFR금리액티브(합성)',       'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '455890', 'name': 'RISE 머니마켓액티브',              'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '469830', 'name': 'SOL 초단기채권액티브',             'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '153130', 'name': 'KODEX 단기채권',                   'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '456610', 'name': 'TIGER 미국달러SOFR금리액티브(합성)','is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '196230', 'name': 'RISE 단기통안채',                  'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '455960', 'name': 'RISE 미국달러SOFR금리액티브(합성)', 'is_risk': False, 'category': 'mmf_ultrashort'},
    {'ticker': '468370', 'name': 'KODEX iShares미국인플레이션국채액티브','is_risk': False,'category': 'mmf_ultrashort'},
    {'ticker': '430500', 'name': 'KIWOOM 물가채KIS',                 'is_risk': False, 'category': 'mmf_ultrashort'},
    # 해외채권_회사채 (안전)
    {'ticker': '458260', 'name': 'TIGER 미국투자등급회사채액티브(H)', 'is_risk': False, 'category': 'overseas_bond_corp'},
    {'ticker': '468380', 'name': 'KODEX iShares미국하이일드액티브',   'is_risk': False, 'category': 'overseas_bond_corp'},
    {'ticker': '455660', 'name': 'ACE 미국하이일드액티브(H)',          'is_risk': False, 'category': 'overseas_bond_corp'},
    {'ticker': '468630', 'name': 'KODEX iShares미국투자등급회사채액티브','is_risk': False,'category': 'overseas_bond_corp'},
    {'ticker': '332620', 'name': 'PLUS 미국장기우량회사채',           'is_risk': False, 'category': 'overseas_bond_corp'},
    {'ticker': '332610', 'name': 'PLUS 미국단기회사채(AAA~A)',        'is_risk': False, 'category': 'overseas_bond_corp'},
    {'ticker': '182490', 'name': 'TIGER 단기선진하이일드(합성 H)',    'is_risk': False, 'category': 'overseas_bond_corp'},
    # 해외채권_종합 (안전)
    {'ticker': '453850', 'name': 'ACE 미국30년국채액티브(H)',              'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '458250', 'name': 'TIGER 미국30년국채스트립액티브(합성 H)', 'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '484790', 'name': 'KODEX 미국30년국채액티브(H)',            'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '329750', 'name': 'TIGER 미국달러단기채권액티브',           'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '472870', 'name': 'RISE 미국30년국채엔화노출(합성 H)',      'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '476760', 'name': 'ACE 미국30년국채액티브',                 'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '305080', 'name': 'TIGER 미국채10년선물',                   'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '308620', 'name': 'KODEX 미국10년국채선물',                 'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '476750', 'name': 'ACE 미국30년국채엔화노출액티브(H)',       'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '489000', 'name': 'PLUS 일본엔화초단기국채(합성)',           'is_risk': False, 'category': 'overseas_bond'},
    {'ticker': '267440', 'name': 'RISE 미국장기국채선물(H)',                'is_risk': False, 'category': 'overseas_bond'},
    # 국내채권_회사채 (안전) — 0061Z0, 0016X0 는 특수 티커
    {'ticker': '273130', 'name': 'KODEX 종합채권(AA-이상)액티브',     'is_risk': False, 'category': 'domestic_bond_corp'},
    {'ticker': '385540', 'name': 'RISE 종합채권(A-이상)액티브',       'is_risk': False, 'category': 'domestic_bond_corp'},
    {'ticker': '0061Z0', 'name': 'RISE 단기특수은행채액티브',          'is_risk': False, 'category': 'domestic_bond_corp', 'special_ticker': True},
    {'ticker': '451540', 'name': 'TIGER 종합채권(AA-이상)액티브',     'is_risk': False, 'category': 'domestic_bond_corp'},
    {'ticker': '438330', 'name': 'TIGER 투자등급회사채액티브',        'is_risk': False, 'category': 'domestic_bond_corp'},
    {'ticker': '0016X0', 'name': 'SOL 중단기회사채(A-이상)액티브',    'is_risk': False, 'category': 'domestic_bond_corp', 'special_ticker': True},
    {'ticker': '363570', 'name': 'KODEX 장기종합채권(AA-이상)액티브', 'is_risk': False, 'category': 'domestic_bond_corp'},
    {'ticker': '278620', 'name': 'PLUS 단기채권액티브',               'is_risk': False, 'category': 'domestic_bond_corp'},
    {'ticker': '336160', 'name': 'RISE 금융채액티브',                 'is_risk': False, 'category': 'domestic_bond_corp'},
    # 국내채권_종합 (안전)
    {'ticker': '157450', 'name': 'TIGER 단기통안채',         'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '114260', 'name': 'KODEX 국고채3년',          'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '148070', 'name': 'KIWOOM 국고채10년',        'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '439870', 'name': 'KODEX 국고채30년액티브',   'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '302190', 'name': 'TIGER 중장기국채',         'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '272560', 'name': 'RISE 단기국공채액티브',    'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '114100', 'name': 'RISE 국고채3년',           'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '451530', 'name': 'TIGER 국고채30년스트립액티브','is_risk': False,'category': 'domestic_bond'},
    {'ticker': '298340', 'name': 'PLUS 국채선물3년',         'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '426150', 'name': 'WON 대한민국국고채액티브', 'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '152380', 'name': 'KODEX 국채선물10년',       'is_risk': False, 'category': 'domestic_bond'},
    {'ticker': '397420', 'name': 'RISE 국채선물5년추종',     'is_risk': False, 'category': 'domestic_bond'},
]

import pandas as pd
df_meta = pd.DataFrame(ETF_LIST)
print(f'총 ETF: {len(df_meta)}개')
print(df_meta.groupby(['category', 'is_risk']).size().to_string())

In [ ]:
# ── 데이터 수집 ───────────────────────────────────────────────────────────────
import FinanceDataReader as fdr
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
import time

START_DATE = '2019-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

# 특수 티커 제외 (0061Z0, 0016X0) — 별도 처리
normal_etfs  = [e for e in ETF_LIST if not e.get('special_ticker', False)]
special_etfs = [e for e in ETF_LIST if e.get('special_ticker', False)]

results = {}
failed  = []

for etf in tqdm(normal_etfs, desc='ETF 수집'):
    ticker = etf['ticker']
    try:
        df = fdr.DataReader(ticker, START_DATE, END_DATE)
        if df.empty:
            raise ValueError('빈 데이터')
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        df.index = pd.to_datetime(df.index)
        df['ticker']   = ticker
        df['name']     = etf['name']
        df['category'] = etf['category']
        df['is_risk']  = etf['is_risk']
        results[ticker] = df
    except Exception as e:
        failed.append({'ticker': ticker, 'name': etf['name'], 'error': str(e)})
    time.sleep(0.1)  # API 요청 간격

print(f'\n성공: {len(results)}개 | 실패: {len(failed)}개')
if failed:
    print('실패 목록:')
    for f in failed:
        print(f"  {f['ticker']} {f['name']}: {f['error']}")

In [ ]:
# ── 특수 티커 수집 (pykrx fallback) ──────────────────────────────────────────
from pykrx import stock as krx

for etf in special_etfs:
    ticker = etf['ticker']
    try:
        df = krx.get_market_ohlcv_by_date(
            START_DATE.replace('-', ''),
            END_DATE.replace('-', ''),
            ticker
        )
        if df.empty:
            raise ValueError('빈 데이터')
        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Change']
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        df.index = pd.to_datetime(df.index)
        df['ticker']   = ticker
        df['name']     = etf['name']
        df['category'] = etf['category']
        df['is_risk']  = etf['is_risk']
        results[ticker] = df
        print(f'특수 티커 수집 성공: {ticker} {etf["name"]}')
    except Exception as e:
        failed.append({'ticker': ticker, 'name': etf['name'], 'error': str(e)})
        print(f'특수 티커 실패: {ticker} — {e}')

In [ ]:
# ── 데이터 품질 검증 ─────────────────────────────────────────────────────────
MIN_ROWS = 250  # 최소 1년치 거래일

quality = []
for ticker, df in results.items():
    quality.append({
        'ticker':    ticker,
        'name':      df['name'].iloc[0],
        'category':  df['category'].iloc[0],
        'rows':      len(df),
        'start':     df.index.min().date(),
        'end':       df.index.max().date(),
        'null_pct':  df[['Open','High','Low','Close','Volume']].isnull().mean().mean() * 100,
        'ok':        len(df) >= MIN_ROWS,
    })

df_quality = pd.DataFrame(quality).sort_values('rows', ascending=False)
print(f'데이터 충분 (>={MIN_ROWS}행): {df_quality["ok"].sum()}개')
print(f'데이터 부족:                  {(~df_quality["ok"]).sum()}개')
print('\n데이터 부족 종목:')
print(df_quality[~df_quality['ok']][['ticker', 'name', 'rows', 'start']].to_string(index=False))

In [ ]:
# ── 기술지표 추가 ─────────────────────────────────────────────────────────────
# 졸업 프로젝트와 동일한 20개 피처 구조 유지
import numpy as np

def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    c = df['Close']
    h = df['High']
    l = df['Low']
    v = df['Volume']

    # 이동평균
    df['SMA_20']     = c.rolling(20).mean()
    df['SMA_60']     = c.rolling(60).mean()
    df['EMA_12']     = c.ewm(span=12, adjust=False).mean()
    df['EMA_26']     = c.ewm(span=26, adjust=False).mean()

    # MACD
    df['MACD']          = df['EMA_12'] - df['EMA_26']
    df['MACD_signal']   = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']     = df['MACD'] - df['MACD_signal']

    # RSI
    delta   = c.diff()
    gain    = delta.clip(lower=0).rolling(14).mean()
    loss    = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI_14'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    # 볼린저 밴드
    mid             = c.rolling(20).mean()
    std             = c.rolling(20).std()
    df['BB_upper']  = mid + 2 * std
    df['BB_lower']  = mid - 2 * std
    df['BB_width']  = (df['BB_upper'] - df['BB_lower']) / mid

    # 거래량
    df['Volume_ratio'] = v / v.rolling(20).mean()

    # 수익률 / 변동성
    df['Return_1d']     = c.pct_change()
    df['Volatility_20d']= df['Return_1d'].rolling(20).std()

    # ATR
    tr = pd.concat([
        h - l,
        (h - c.shift()).abs(),
        (l - c.shift()).abs()
    ], axis=1).max(axis=1)
    df['ATR_14'] = tr.rolling(14).mean()

    return df

processed = {}
for ticker, df in tqdm(results.items(), desc='지표 계산'):
    processed[ticker] = add_indicators(df)

print(f'처리 완료: {len(processed)}개')
print('피처 목록:', [c for c in processed[list(processed.keys())[0]].columns])

In [ ]:
# ── 저장 ─────────────────────────────────────────────────────────────────────
import os

# 1) Long format — 모든 ETF 합치기 (분석/모델 학습용)
df_long = pd.concat(processed.values(), axis=0)
df_long.index.name = 'Date'
df_long.reset_index(inplace=True)
long_path = os.path.join(DRIVE_PATH, 'etf_ohlcv_indicators.parquet')
df_long.to_parquet(long_path, index=False)
print(f'Long format 저장: {long_path}  shape={df_long.shape}')

# 2) Wide format — 종가만, 날짜 x 티커 (포트폴리오 분석용)
df_close = pd.DataFrame({
    t: df.set_index('Date')['Close'] if 'Date' in df.columns
       else df['Close']
    for t, df in processed.items()
})
close_path = os.path.join(DRIVE_PATH, 'etf_close_wide.parquet')
df_close.to_parquet(close_path)
print(f'Wide format 저장: {close_path}  shape={df_close.shape}')

# 3) 메타데이터
meta_path = os.path.join(DRIVE_PATH, 'etf_meta.parquet')
df_quality.to_parquet(meta_path, index=False)
print(f'메타데이터 저장: {meta_path}')

print('\n모든 파일 저장 완료!')

In [ ]:
# ── 빠른 시각화 확인 ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트

# 카테고리별 ETF 수
cat_cnt = df_long.groupby('category')['ticker'].nunique().sort_values(ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_cnt.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('카테고리별 ETF 수')
axes[0].set_xlabel('종목 수')

# 주요 ETF 종가 추이 (카테고리 대표 1개씩)
rep = {
    '미국S&P500':  '360750',
    'KODEX200':   '069500',
    '미국30년채':  '453850',
    'CD금리':     '459580',
    '금현물':     '411060',
}
for label, t in rep.items():
    if t in processed:
        s = processed[t].set_index('Date')['Close'] if 'Date' in processed[t].columns \
            else processed[t]['Close']
        (s / s.iloc[0] * 100).plot(ax=axes[1], label=label)

axes[1].set_title('주요 ETF 정규화 종가 (기준=100)')
axes[1].legend(fontsize=8)
axes[1].set_ylabel('정규화 가격')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, 'etf_overview.png'), dpi=120)
plt.show()